In [2]:
import pandas as pd, numpy as np
train = pd.read_csv("../data/train_clean.csv")

def woe_table(data, col, target="TARGET", bins=10):
    x = data[col]
    if pd.api.types.is_numeric_dtype(x) and x.nunique() > bins:
        groups = pd.qcut(x, q=bins, duplicates="drop")
        groups = groups.cat.add_categories("MISSING").fillna("MISSING")
    else:
        groups = x.astype(str).where(x.notna(), "MISSING")
    t = data.groupby(groups, observed=True)[target].agg(total="count", bad="sum")
    t["good"] = t["total"] - t["bad"]
    t["dist_good"] = (t["good"] + 0.5) / t["good"].sum()
    t["dist_bad"] = (t["bad"] + 0.5) / t["bad"].sum()
    t["bad_rate"] = t["bad"] / t["total"]
    t["woe"] = np.log(t["dist_good"] / t["dist_bad"])
    t["iv"] = (t["dist_good"] - t["dist_bad"]) * t["woe"]
    return t

features = [c for c in train.columns if c not in ["SK_ID_CURR", "TARGET"]]
iv = pd.Series({c: woe_table(train, c)["iv"].sum() for c in features}).sort_values(ascending=False)
print(iv.head(20).round(3))
print(woe_table(train, "EXT_SOURCE_3")[["total", "bad_rate", "woe"]].round(3))

EXT_SOURCE_3                   0.324
EXT_SOURCE_2                   0.309
EXT_SOURCE_1                   0.155
DAYS_EMPLOYED                  0.111
YEARS_EMPLOYED                 0.111
AMT_GOODS_PRICE                0.091
DAYS_BIRTH                     0.086
AGE_YEARS                      0.086
OCCUPATION_TYPE                0.083
ORGANIZATION_TYPE              0.072
NAME_INCOME_TYPE               0.054
REGION_RATING_CLIENT_W_CITY    0.052
NAME_EDUCATION_TYPE            0.050
REGION_RATING_CLIENT           0.049
AMT_CREDIT                     0.046
DAYS_LAST_PHONE_CHANGE         0.044
CODE_GENDER                    0.039
DAYS_ID_PUBLISH                0.037
FLOORSMAX_AVG                  0.036
REGION_POPULATION_RELATIVE     0.035
dtype: float64
                    total  bad_rate    woe
EXT_SOURCE_3                              
(-0.000473, 0.228]  17358     0.197 -1.026
(0.228, 0.33]       17318     0.127 -0.502
(0.33, 0.407]       17162     0.098 -0.211
(0.407, 0.474]      17383     

In [3]:
exclude = ["DAYS_BIRTH", "DAYS_EMPLOYED", "CODE_GENDER"]
candidates = [c for c in iv.index if iv[c] >= 0.02 and c not in exclude]
print(len(candidates))
print(candidates)

32
['EXT_SOURCE_3', 'EXT_SOURCE_2', 'EXT_SOURCE_1', 'YEARS_EMPLOYED', 'AMT_GOODS_PRICE', 'AGE_YEARS', 'OCCUPATION_TYPE', 'ORGANIZATION_TYPE', 'NAME_INCOME_TYPE', 'REGION_RATING_CLIENT_W_CITY', 'NAME_EDUCATION_TYPE', 'REGION_RATING_CLIENT', 'AMT_CREDIT', 'DAYS_LAST_PHONE_CHANGE', 'DAYS_ID_PUBLISH', 'FLOORSMAX_AVG', 'REGION_POPULATION_RELATIVE', 'EMPLOYED_ANOM', 'FLAG_EMP_PHONE', 'REG_CITY_NOT_WORK_CITY', 'LIVINGAREA_AVG', 'APARTMENTS_AVG', 'ELEVATORS_AVG', 'FLAG_DOCUMENT_3', 'DAYS_REGISTRATION', 'YEARS_BEGINEXPLUATATION_AVG', 'AMT_ANNUITY', 'ENTRANCES_AVG', 'OWN_CAR_AGE', 'NAME_FAMILY_STATUS', 'REG_CITY_NOT_LIVE_CITY', 'NONLIVINGAREA_AVG']


In [4]:
def fit_woe(data, col, target="TARGET", bins=10):
    x = data[col]
    if pd.api.types.is_numeric_dtype(x) and x.nunique() > bins:
        _, edges = pd.qcut(x, q=bins, duplicates="drop", retbins=True)
        edges[0], edges[-1] = -np.inf, np.inf
        groups = pd.cut(x, edges).astype(str).where(x.notna(), "MISSING")
    else:
        edges = None
        groups = x.astype(str).where(x.notna(), "MISSING")
    t = data.groupby(groups)[target].agg(total="count", bad="sum")
    good = t["total"] - t["bad"]
    woe = np.log(((good + 0.5) / good.sum()) / ((t["bad"] + 0.5) / t["bad"].sum()))
    return {"edges": edges, "woe": woe.to_dict()}

def apply_woe(data, col, rule):
    x = data[col]
    if rule["edges"] is not None:
        groups = pd.cut(x, rule["edges"]).astype(str).where(x.notna(), "MISSING")
    else:
        groups = x.astype(str).where(x.notna(), "MISSING")
    return groups.map(rule["woe"]).fillna(0)

In [5]:
rules = {c: fit_woe(train, c) for c in candidates}
train_woe = pd.DataFrame({c: apply_woe(train, c, rules[c]) for c in candidates})

corr = train_woe.corr().abs()
selected = []
for c in candidates:                      # ไล่จาก IV สูงไปต่ำ
    if all(corr.loc[c, s] < 0.7 for s in selected):
        selected.append(c)                # เก็บไว้ถ้าไม่ซ้ำกับตัวที่เลือกแล้ว
print(len(selected), selected)

23 ['EXT_SOURCE_3', 'EXT_SOURCE_2', 'EXT_SOURCE_1', 'YEARS_EMPLOYED', 'AMT_GOODS_PRICE', 'AGE_YEARS', 'OCCUPATION_TYPE', 'ORGANIZATION_TYPE', 'NAME_INCOME_TYPE', 'REGION_RATING_CLIENT_W_CITY', 'NAME_EDUCATION_TYPE', 'AMT_CREDIT', 'DAYS_LAST_PHONE_CHANGE', 'DAYS_ID_PUBLISH', 'FLOORSMAX_AVG', 'REGION_POPULATION_RELATIVE', 'REG_CITY_NOT_WORK_CITY', 'FLAG_DOCUMENT_3', 'DAYS_REGISTRATION', 'AMT_ANNUITY', 'OWN_CAR_AGE', 'NAME_FAMILY_STATUS', 'REG_CITY_NOT_LIVE_CITY']


In [6]:
from sklearn.linear_model import LogisticRegression

feats = selected.copy()
while True:
    model = LogisticRegression(max_iter=1000).fit(train_woe[feats], train["TARGET"])
    coef = pd.Series(model.coef_[0], index=feats)
    if (coef < 0).all():
        break
    worst = coef.idxmax()
    print("drop", worst, round(coef[worst], 3))
    feats.remove(worst)

print(len(feats), "variables")
coef.sort_values().round(3)

drop AGE_YEARS 0.229
drop NAME_INCOME_TYPE 0.144
drop AMT_CREDIT 0.143
drop REG_CITY_NOT_WORK_CITY 0.096
drop REGION_POPULATION_RELATIVE 0.005
18 variables


EXT_SOURCE_3                  -0.824
EXT_SOURCE_2                  -0.730
FLAG_DOCUMENT_3               -0.580
AMT_GOODS_PRICE               -0.548
NAME_EDUCATION_TYPE           -0.542
OWN_CAR_AGE                   -0.539
EXT_SOURCE_1                  -0.510
ORGANIZATION_TYPE             -0.393
AMT_ANNUITY                   -0.306
REGION_RATING_CLIENT_W_CITY   -0.301
FLOORSMAX_AVG                 -0.289
OCCUPATION_TYPE               -0.274
YEARS_EMPLOYED                -0.272
DAYS_REGISTRATION             -0.227
REG_CITY_NOT_LIVE_CITY        -0.208
NAME_FAMILY_STATUS            -0.198
DAYS_LAST_PHONE_CHANGE        -0.191
DAYS_ID_PUBLISH               -0.153
dtype: float64

In [7]:
from sklearn.metrics import roc_auc_score
test = pd.read_csv("../data/test_clean.csv")
test_woe = pd.DataFrame({c: apply_woe(test, c, rules[c]) for c in feats})
auc_train = roc_auc_score(train["TARGET"], model.predict_proba(train_woe[feats])[:, 1])
auc_test = roc_auc_score(test["TARGET"], model.predict_proba(test_woe[feats])[:, 1])
print(f"Train AUC {auc_train:.4f} | Test AUC {auc_test:.4f} | Test Gini {2*auc_test-1:.4f}")

Train AUC 0.7418 | Test AUC 0.7423 | Test Gini 0.4847


In [8]:
PDO, BASE_SCORE, BASE_ODDS = 20, 600, 50
factor = PDO / np.log(2)
offset = BASE_SCORE - factor * np.log(BASE_ODDS)
n = len(feats)
b0 = model.intercept_[0]

rows = []
for c in feats:
    for grp, w in rules[c]["woe"].items():
        pts = -(coef[c] * w + b0 / n) * factor + offset / n
        rows.append({"variable": c, "bin": grp, "woe": round(w, 3), "points": round(pts)})
scorecard = pd.DataFrame(rows)
scorecard.to_csv("../reports/scorecard.csv", index=False)
scorecard[scorecard["variable"] == "EXT_SOURCE_3"]

,variable,bin,woe,points
0,EXT_SOURCE_3,"(-inf, 0.228]",-1.026,7
1,EXT_SOURCE_3,"(0.228, 0.33]",-0.502,19
2,EXT_SOURCE_3,"(0.33, 0.407]",-0.211,26
3,EXT_SOURCE_3,"(0.407, 0.474]",0.016,31
4,EXT_SOURCE_3,"(0.474, 0.535]",0.288,38
5,EXT_SOURCE_3,"(0.535, 0.59]",0.435,41
6,EXT_SOURCE_3,"(0.59, 0.641]",0.505,43
7,EXT_SOURCE_3,"(0.641, 0.694]",0.729,48
8,EXT_SOURCE_3,"(0.694, 0.749]",0.900,52
9,EXT_SOURCE_3,"(0.749, inf]",0.957,54


In [9]:
def score(data):
    total = pd.Series(0.0, index=data.index)
    for c in feats:
        w = apply_woe(data, c, rules[c])
        total += -(coef[c] * w + b0 / n) * factor + offset / n
    return total.round()

test["SCORE"] = score(test)
print(test["SCORE"].describe())
test.groupby(pd.qcut(test["SCORE"], 10), observed=True)["TARGET"].agg(["count", "mean"]).round(3)

count    92253.000000
mean       567.386600
std         27.043706
min        469.000000
25%        549.000000
50%        568.000000
75%        586.000000
max        687.000000
Name: SCORE, dtype: float64


,count,mean
SCORE,,
"(468.999, 532.0]",9621,0.248
"(532.0, 544.0]",9135,0.145
"(544.0, 553.0]",9549,0.107
"(553.0, 561.0]",9714,0.079
"(561.0, 568.0]",9022,0.063
"(568.0, 575.0]",9229,0.048
"(575.0, 582.0]",8611,0.037
"(582.0, 591.0]",9655,0.030
"(591.0, 602.0]",8766,0.022


In [10]:
kaggle_test = pd.read_csv("../data/application_test.csv")

# ต้อง clean แบบเดียวกับ train
kaggle_test["EMPLOYED_ANOM"] = (kaggle_test["DAYS_EMPLOYED"] == 365243).astype(int)
kaggle_test.loc[kaggle_test["EMPLOYED_ANOM"] == 1, "DAYS_EMPLOYED"] = None
kaggle_test["AGE_YEARS"] = -kaggle_test["DAYS_BIRTH"] / 365
kaggle_test["YEARS_EMPLOYED"] = -kaggle_test["DAYS_EMPLOYED"] / 365

# แปลงเป็น WoE ด้วย rules จาก train แล้วทำนาย PD
X_kaggle = pd.DataFrame({c: apply_woe(kaggle_test, c, rules[c]) for c in feats})
submission = pd.DataFrame({
    "SK_ID_CURR": kaggle_test["SK_ID_CURR"],
    "TARGET": model.predict_proba(X_kaggle)[:, 1],   # ส่งเป็น "ความน่าจะเป็น" ไม่ใช่ 0/1
})
submission.to_csv("../reports/submission_scorecard.csv", index=False)
print(submission.shape, submission.isnull().sum().sum())

(48744, 2) 0
